# Notebook 0: first exploration of MIMIC-IV (outputs cleared: they displayed raw records)

In [ ]:
%pip install seaborn matplotlib pandas scikit-learn jupyter 

In [ ]:
%pip install "setuptools<81"

In [ ]:
# ============================================================
# Imports
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Paths
ROOT = Path('..')
DATA = ROOT / 'data' / 'hosp'
PROCESSED = ROOT / 'processed'
FIGURES = ROOT / 'figures'
RESULTS = ROOT / 'results'

# Create output dirs if needed
for d in [PROCESSED, FIGURES, RESULTS]:
    d.mkdir(parents=True, exist_ok=True)

print("Imports OK")
print(f"Data path: {DATA.resolve()}")

In [ ]:
# ============================================================
# Load data
# ============================================================
admissions = pd.read_csv(DATA / 'admissions.csv.gz', compression='gzip')
diagnoses  = pd.read_csv(DATA / 'diagnoses_icd.csv.gz', compression='gzip')
patients   = pd.read_csv(DATA / 'patients.csv.gz', compression='gzip')

print(f"admissions : {admissions.shape[0]:>10,} rows × {admissions.shape[1]} cols")
print(f"diagnoses  : {diagnoses.shape[0]:>10,} rows × {diagnoses.shape[1]} cols")
print(f"patients   : {patients.shape[0]:>10,} rows × {patients.shape[1]} cols")

In [ ]:
# ============================================================
# Preview
# ============================================================
print("=== ADMISSIONS ===")
display(admissions.head())

print("\n=== DIAGNOSES_ICD ===")
display(diagnoses.head())

print("\n=== PATIENTS ===")
display(patients.head())

In [ ]:
#PyHealth to implement the transformer

from pyhealth.datasets import MIMIC4Dataset
from pyhealth.models import Transformer
from pyhealth.tasks import drug_recommendation_mimic4_fn

In [ ]:
# ICD version distribution
print("ICD version distribution in diagnoses:")
print(diagnoses['icd_version'].value_counts())

In [ ]:
# ============================================================
# Load and preprocess manually — ICD-10 level-3 (primary)
# ============================================================
import pandas as pd
from pathlib import Path

DATA = Path('../data/hosp')

admissions = pd.read_csv(DATA / 'admissions.csv.gz')
diagnoses  = pd.read_csv(DATA / 'diagnoses_icd.csv.gz')
patients   = pd.read_csv(DATA / 'patients.csv.gz')

# ICD-10 only
diagnoses = diagnoses[diagnoses['icd_version'] == 10].copy()

# Truncate to level-3 (E11.2 → E11)
diagnoses['icd_code_l3'] = diagnoses['icd_code'].str[:3]

print(f"ICD-10 diagnoses: {diagnoses.shape[0]:,} rows")
print(f"Unique level-3 codes: {diagnoses['icd_code_l3'].nunique()}")

In [ ]:
# ============================================================
# Manual CCS mapping — download from MedCo's GitHub directly
# ============================================================
import subprocess

# Clone MedCo repo to get their preprocessing code and mapping files
subprocess.run([
    "git", "clone", 
    "https://github.com/mohsen-nyb/MedCo.git",
    "../MedCo"
], check=True)

# Check what preprocessing files they have
import os
for root, dirs, files in os.walk("../MedCo"):
    for f in files:
        if any(ext in f for ext in ['.py', '.csv', '.json']):
            print(os.path.join(root, f))

In [ ]:
GEM_DX = ROOT / 'data' / 'mappings' / '2018_I10gem.txt'
GEM_PX = ROOT / 'data' / 'mappings' / 'gem_pcsi9.txt'